# 🎬 CFX Studios — Cloud Video Compressor (1-by-1 Mode)
### Compress any specific video under 700MB in Google Drive without downloading to your PC.

---
### ⚡ Features:
- **No Batch Processing**: You choose exactly which single video you want to compress, one at a time.
- **0 MB Downloaded to PC**: Runs in Google's cloud server at 1,000+ Mbps.
- **T4 GPU Hardware Accelerated**: Re-encodes in 1–2 minutes with `hevc_nvenc`.
- **In-Colab Video Preview**: Watch the compressed video right inside Colab to check quality before replacing!

In [ ]:
#@title 🚀 Step 1: Connect Google Drive & Detect GPU { display-mode: "form" }
import os
import sys
import shutil
import subprocess
import time

print("🔄 [1/2] Connecting Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n⚡ [2/2] Checking GPU & Hardware Encoder...")
gpu_available = False
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
    if gpu_info:
        print(f"🔥 GPU Detected: {gpu_info} (Ultra-Fast Hardware Encoding ACTIVE)")
        gpu_available = True
except Exception:
    print("ℹ️ No GPU detected. Running on CPU (libx265).")
    print("💡 Tip: Go to 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU' for 5x faster speeds!")

print("\n✨ Ready! Proceed to Step 2.")


In [ ]:
#@title 🔍 Step 2: List Large Videos & Choose Which One to Compress { display-mode: "form" }
#@markdown Folder to scan (or leave blank to auto-detect 'MediaFire_Transfers'):
FOLDER_PATH = "" #@param {type:"string"}
MIN_SIZE_MB = 700 #@param {type:"integer"}

import os

if not FOLDER_PATH.strip():
    candidates = [
        "/content/drive/MyDrive/MediaFire_Transfers",
        "/content/drive/Shareddrives",
        "/content/drive/MyDrive"
    ]
    found = None
    for cand in candidates:
        if os.path.exists(cand):
            if cand.endswith("MediaFire_Transfers"):
                found = cand
                break
            for root, dirs, _ in os.walk(cand):
                if "MediaFire_Transfers" in dirs:
                    found = os.path.join(root, "MediaFire_Transfers")
                    break
            if found: break
    FOLDER_PATH = found if found else "/content/drive/MyDrive"

print(f"📂 Scanning: {FOLDER_PATH}\n")

video_extensions = ('.mp4', '.mkv', '.mov', '.avi', '.webm')
scanned_files = []

for root, dirs, files in os.walk(FOLDER_PATH):
    for f in files:
        if f.lower().endswith(video_extensions) and not f.startswith(('._', 'temp_')):
            full_path = os.path.join(root, f)
            try:
                sz = os.path.getsize(full_path)
                sz_mb = sz / (1024 * 1024)
                if sz_mb >= MIN_SIZE_MB:
                    rel_folder = os.path.relpath(root, FOLDER_PATH)
                    scanned_files.append({
                        'name': f,
                        'path': full_path,
                        'folder': rel_folder if rel_folder != '.' else 'Root',
                        'size_mb': sz_mb,
                        'size_gb': sz_mb / 1024.0
                    })
            except Exception:
                pass

scanned_files.sort(key=lambda x: x['size_mb'], reverse=True)

print("=" * 105)
print(f"{ '#':<4} | { 'FOLDER':<28} | { 'FILE NAME':<40} | { 'SIZE':<15}")
print("=" * 105)
for i, item in enumerate(scanned_files, 1):
    f_display = (item['folder'][:26] + '..') if len(item['folder']) > 26 else item['folder']
    n_display = (item['name'][:38] + '..') if len(item['name']) > 38 else item['name']
    sz_display = f"{item['size_gb']:.2f} GB ({int(item['size_mb'])} MB)"
    print(f"{i:<4} | {f_display:<28} | {n_display:<40} | {sz_display:<15}")
print("=" * 105)

print(f"\n👉 Choose any video above by entering its number (1 to {len(scanned_files)}) in Step 3!")


In [ ]:
#@title ⚡ Step 3: Compress Selected Video (With Live Realtime Progress Bar) { display-mode: "form" }
#@markdown Enter the number (#) of the video you want to compress from Step 2:
VIDEO_NUMBER = 1 #@param {type:"integer"}
#@markdown Target max file size:
TARGET_MAX_MB = 680 #@param {type:"integer"}
CODEC = "H.265 / HEVC (Recommended - Best Quality)" #@param ["H.265 / HEVC (Recommended - Best Quality)", "H.264 (Universal)"]
#@markdown Save mode:
ACTION = "Direct In-Place Replacement (Keeps Drive Link)" #@param ["Direct In-Place Replacement (Keeps Drive Link)", "Save New File Alongside (e.g. name_compressed.mp4)", "Only Preview in Colab (Do Not Touch Drive)"]

import subprocess
import time
import shutil
from tqdm.notebook import tqdm
from IPython.display import display, HTML

if 'scanned_files' not in globals() or not scanned_files:
    print("❌ Please run Step 2 first to scan the folder!")
    sys.exit(1)

if VIDEO_NUMBER < 1 or VIDEO_NUMBER > len(scanned_files):
    print(f"❌ Invalid number! Please enter a number between 1 and {len(scanned_files)}.")
    sys.exit(1)

target = scanned_files[VIDEO_NUMBER - 1]
orig_path = target['path']
orig_name = target['name']
orig_size_mb = target['size_mb']

print(f"🎬 Selected Video #{VIDEO_NUMBER}: {orig_name}")
print(f"📁 Folder: {target['folder']}")
print(f"⚖️ Current Size: {orig_size_mb:.1f} MB ({target['size_gb']:.2f} GB)")
print(f"🎯 Target Ceiling: < {TARGET_MAX_MB} MB\n")

# Get video duration
def get_duration(path):
    try:
        cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", path]
        return float(subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode().strip())
    except Exception:
        return None

duration = get_duration(orig_path)
audio_bitrate_kbps = 160
if duration and duration > 0:
    target_total_kbps = (TARGET_MAX_MB * 8192 * 0.95) / duration
    target_video_kbps = max(500, int(target_total_kbps - audio_bitrate_kbps))
    print(f"⏱️ Duration: {int(duration // 60)}m {int(duration % 60)}s | Auto-Target Bitrate: ~{target_video_kbps} kbps")
else:
    duration = 3600 # fallback 1 hr
    target_video_kbps = 2400

TEMP_OUT = "/content/preview_compressed.mp4"
if os.path.exists(TEMP_OUT):
    try: os.remove(TEMP_OUT)
    except: pass

use_hevc = "H.265" in CODEC
if gpu_available:
    vcodec = "hevc_nvenc" if use_hevc else "h264_nvenc"
    cmd = [
        "ffmpeg", "-y", "-hwaccel", "cuda",
        "-i", orig_path,
        "-c:v", vcodec, "-preset", "p5",
        "-b:v", f"{target_video_kbps}k",
        "-maxrate", f"{int(target_video_kbps * 1.3)}k",
        "-bufsize", f"{target_video_kbps * 2}k",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", f"{audio_bitrate_kbps}k",
        "-movflags", "+faststart",
        "-progress", "pipe:1", "-nostats",
        TEMP_OUT
    ]
else:
    vcodec = "libx265" if use_hevc else "libx264"
    cmd = [
        "ffmpeg", "-y",
        "-i", orig_path,
        "-c:v", vcodec, "-preset", "medium",
        "-b:v", f"{target_video_kbps}k",
        "-maxrate", f"{int(target_video_kbps * 1.3)}k",
        "-bufsize", f"{target_video_kbps * 2}k",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", f"{audio_bitrate_kbps}k",
        "-movflags", "+faststart",
        "-progress", "pipe:1", "-nostats",
        TEMP_OUT
    ]

print("\n⚡ Live Encoding Progress:")
pbar = tqdm(total=int(duration), unit="s", desc="🎬 Compressing")
last_sec = 0
current_speed = ""
current_fps = ""

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, universal_newlines=True)

for line in proc.stdout:
    line = line.strip()
    if '=' in line:
        k, v = line.split('=', 1)
        if k == 'out_time_us':
            try:
                curr_sec = int(v) / 1000000.0
                delta = curr_sec - last_sec
                if delta > 0:
                    pbar.update(min(delta, duration - last_sec))
                    last_sec = curr_sec
            except Exception:
                pass
        elif k == 'speed':
            current_speed = v.strip()
        elif k == 'fps':
            current_fps = v.strip()
        pbar.set_postfix({'Speed': current_speed, 'FPS': current_fps})

proc.wait()
if last_sec < duration:
    pbar.update(duration - last_sec)
pbar.close()
elapsed = time.time() - t0

if proc.returncode == 0 and os.path.exists(TEMP_OUT):
    new_size_bytes = os.path.getsize(TEMP_OUT)
    new_size_mb = new_size_bytes / (1024 * 1024)
    saved_mb = orig_size_mb - new_size_mb
    reduction_pct = (saved_mb / orig_size_mb) * 100
    
    print(f"\n✅ Compression Completed in {elapsed:.1f}s!")
    print(f"  • Original Size: {orig_size_mb:.1f} MB ({target['size_gb']:.2f} GB)")
    print(f"  • New Compressed Size: {new_size_mb:.1f} MB")
    print(f"  • Reduction: -{reduction_pct:.1f}% (Saved {saved_mb:.1f} MB!)\n")
    
    if "Direct In-Place" in ACTION:
        shutil.copy2(TEMP_OUT, orig_path)
        print(f"🔄 Successfully updated original file in Google Drive! (Drive link preserved)")
    elif "Alongside" in ACTION:
        base, ext = os.path.splitext(orig_path)
        new_path = f"{base}_compressed{ext}"
        shutil.copy2(TEMP_OUT, new_path)
        print(f"💾 Saved new compressed copy alongside original in Drive: {os.path.basename(new_path)}")
    else:
        print(f"ℹ️ Preview mode only: Drive files were not modified.")
        
    # Show quick preview player
    print("\n📺 Previewing compressed video:")
    display(HTML(f'''
        <video width="640" height="360" controls style="border-radius: 8px; border: 1px solid #444;">
          <source src="/content/preview_compressed.mp4" type="video/mp4">
          Your browser does not support the video tag.
        </video>
    '''))
else:
    print(f"❌ Compression error occurred.")
